# CSE 164 First Segmentation Baseline\n
\n
This notebook runs the first end-to-end supervised segmentation baseline in Google Colab.\n
\n
Expected Drive layout:\n
\n
```text\n
/content/drive/MyDrive/CSE164FinalProject/raw/\n
|-- metadata/\n
|-- test/\n
|-- train_labeled/\n
|-- train_seg/\n
|-- train_unlabeled/\n
`-- val/\n
```\n
\n
All model outputs are written to Drive under `/content/drive/MyDrive/CSE164FinalProject/outputs/`, so checkpoints, figures, and `submission.csv` survive the Colab runtime.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Unzip the datset

In [ ]:
!mkdir -p /content/data

!cp /content/drive/MyDrive/CSE164FinalProject/raw.zip /content/raw.zip

!unzip -q /content/raw.zip -d /content/data

!ls /content/data
!ls /content/data/raw

## Configuration\n
\n
Change these values before running if your Drive folder or training settings are different.

In [2]:
from pathlib import Path

GITHUB_REPO = 'https://github.com/007deepaks/CSE-164FinalProject.git'
REPO_DIR = Path('/content/CSE-164FinalProject')

DRIVE_PROJECT = Path('/content/drive/MyDrive/CSE164FinalProject')
DATA_ROOT = Path('/content/data/raw')
OUTPUT_ROOT = DRIVE_PROJECT / 'outputs'

CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
SUBMISSION_DIR = OUTPUT_ROOT / 'submissions'
PREDICTION_DIR = OUTPUT_ROOT / 'predictions'

EPOCHS = 10
BATCH_SIZE = 8
BASE_CHANNELS = 32
IMAGE_SIZE = 256
NUM_WORKERS = 2

# Set these for a quick Colab sanity check before full training.
MAX_TRAIN_SAMPLES = None
MAX_VAL_SAMPLES = None

for path in [CHECKPOINT_DIR, FIGURE_DIR, SUBMISSION_DIR, PREDICTION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('Data root:', DATA_ROOT)
print('Outputs:', OUTPUT_ROOT)

Data root: /content/drive/MyDrive/CSE164FinalProject/raw
Outputs: /content/drive/MyDrive/CSE164FinalProject/outputs


## Clone or Update Repository\n
\n
If the repo is private and this cell fails, make the repo public temporarily or clone with a GitHub token/Colab GitHub auth. Do not hard-code a token into this notebook before committing.

In [5]:
import getpass

token = getpass.getpass("GitHub Token: ")

if REPO_DIR.exists():
    %cd {REPO_DIR}
    !git pull
else:
    %cd /content
    !git clone https://{token}@github.com/007deepaks/CSE-164FinalProject.git
    %cd {REPO_DIR}

!git status --short

GitHub Token: ··········
/content
Cloning into 'CSE-164FinalProject'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 79 (delta 13), reused 75 (delta 12), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 2.09 MiB | 5.62 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/CSE-164FinalProject


## Install Light Dependencies\n
\n
Colab normally includes CUDA-enabled PyTorch already, so this avoids reinstalling `torch` from `requirements.txt` and only ensures the lightweight packages are available.

In [6]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('gpu:', torch.cuda.get_device_name(0))

!python -m pip install -q numpy pandas pillow

torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4


## Verify Dataset Layout\n
\n
This should show `metadata`, `test`, `train_labeled`, `train_seg`, `train_unlabeled`, and `val` inside `raw/`.

In [ ]:
assert DATA_ROOT.exists(), f'Missing DATA_ROOT: {DATA_ROOT}'

!find {DATA_ROOT} -maxdepth 2 -type d | sort | head -40

!python -m src.data.inspect_dataset \
    --data-root {DATA_ROOT} \
    --sample-masks 3 \
    --frequency-masks 25 \
    --random-seed 164 \
    --pair-check-limit 100

## Train Segmentation Baseline\n
\n
For a quick sanity run, set `MAX_TRAIN_SAMPLES = 16`, `MAX_VAL_SAMPLES = 8`, and `EPOCHS = 1` in the config cell. For the first real baseline, leave both max sample values as `None`.

In [11]:
import subprocess

train_cmd = [
    'python', '-m', 'src.training.train_segmentation',
    '--data-root', str(DATA_ROOT),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--base-channels', str(BASE_CHANNELS),
    '--image-size', str(IMAGE_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--checkpoint-dir', str(CHECKPOINT_DIR),
    '--figure-dir', str(FIGURE_DIR),
]

if MAX_TRAIN_SAMPLES is not None:
    train_cmd += ['--max-train-samples', str(MAX_TRAIN_SAMPLES)]

if MAX_VAL_SAMPLES is not None:
    train_cmd += ['--max-val-samples', str(MAX_VAL_SAMPLES)]

print(' '.join(train_cmd))
subprocess.run(train_cmd, check=True)

python -m src.training.train_segmentation --data-root /content/drive/MyDrive/CSE164FinalProject/raw --epochs 1 --batch-size 4 --base-channels 16 --image-size 256 --num-workers 0 --checkpoint-dir /content/drive/MyDrive/CSE164FinalProject/outputs/checkpoints --figure-dir /content/drive/MyDrive/CSE164FinalProject/outputs/figures


KeyboardInterrupt: 

## Evaluate Best Checkpoint\n
\n
This writes validation prediction panels to the Drive figures folder.

In [9]:
BEST_CHECKPOINT = CHECKPOINT_DIR / 'best_segmentation.pt'
assert BEST_CHECKPOINT.exists(), f'Missing checkpoint: {BEST_CHECKPOINT}'

import subprocess

eval_cmd = [
    'python', '-m', 'src.training.evaluate_segmentation',
    '--checkpoint', str(BEST_CHECKPOINT),
    '--data-root', str(DATA_ROOT),
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', str(BATCH_SIZE),
    '--num-workers', str(NUM_WORKERS),
    '--figure-dir', str(FIGURE_DIR),
]

print(' '.join(eval_cmd))
subprocess.run(eval_cmd, check=True)

AssertionError: Missing checkpoint: /content/drive/MyDrive/CSE164FinalProject/outputs/checkpoints/best_segmentation.pt

## Generate Kaggle Submission\n
\n
The final CSV is written to Drive at `CSE164FinalProject/outputs/submissions/submission.csv`.

In [ ]:
SUBMISSION_CSV = SUBMISSION_DIR / 'submission.csv'

import subprocess

predict_cmd = [
    'python', '-m', 'src.training.predict_test',
    '--checkpoint', str(BEST_CHECKPOINT),
    '--data-root', str(DATA_ROOT),
    '--output', str(SUBMISSION_CSV),
    '--image-size', str(IMAGE_SIZE),
    '--batch-size', '1',
    '--num-workers', str(NUM_WORKERS),
]

print(' '.join(predict_cmd))
subprocess.run(predict_cmd, check=True)

print('Submission stored in Drive at:', SUBMISSION_CSV)

## Inspect Output Files

In [ ]:
import pandas as pd

print('Checkpoints:')
for path in sorted(CHECKPOINT_DIR.glob('*')):
    print(path.name, f'{path.stat().st_size / (1024 * 1024):.1f} MB')

print('\nRecent figures:')
for path in sorted(FIGURE_DIR.glob('*'))[-10:]:
    print(path.name)

print('\nSubmissions:')
for path in sorted(SUBMISSION_DIR.glob('*.csv')):
    print(path.name, f'{path.stat().st_size / (1024 * 1024):.1f} MB')

df = pd.read_csv(SUBMISSION_CSV)

print('\nSubmission preview:')
print(df.head())

print('rows:', len(df))
print('columns:', list(df.columns))